# Build Model

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime as dt
import bentoml
from joblib import load
import os


pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from dota_oracle_common.postgresql import DatabaseEngineFactory
from sqlalchemy.ext.asyncio import AsyncSession

engine = DatabaseEngineFactory.get_engine()
engine

In [ ]:
from dota_oracle_common.repositories.match_repository import MatchRepository


async with AsyncSession(engine) as session:
    match_repo = MatchRepository(session)
    
    matches = await match_repo.get_match_details(
        relationship_fields=[
            "outcome", "team_features", "player_hero_features", "hero_features"
        ]
    )
    
    print(f"number of matches: {len(matches)}")
    



dota_oracle_common.repositories.base_repository - Retrieved 96821 records for MatchTable
dota_oracle_common.repositories.match_repository - Found 96821 MatchTable details.


number of matches: 96821


In [9]:
match_outcome_list = []
features_list = []

for match in matches:
    outcome = match.outcome
    match_outcome_list.append(outcome.model_dump())
    
    team_features = match.team_features
    hero_features = match.hero_features
    player_hero_features = match.player_hero_features
    
    features_dictionary = {**team_features.model_dump(), **hero_features.model_dump(), **player_hero_features.model_dump()}
    
    features_list.append(features_dictionary)
    

print(f"count match_outcome_list: {len(match_outcome_list)}")
print(f"count features_list: {len(features_list)}")

count match_outcome_list: 96821
count features_list: 96821


In [10]:
outcome_df = pd.DataFrame(match_outcome_list)
features_df = pd.DataFrame(features_list)

In [11]:
outcome_df

,match_id,radiant_win
0,8230722475,True
1,8230701740,False
2,8230693148,True
3,8230677659,True
4,8230656847,True
...,...,...
96816,5999283181,False
96817,5999249937,False
96818,5999214195,False
96819,5999201501,True


In [12]:
features_df

,dire_win_rate,radiant_win_rate,match_id,radiant_dire_matchup,hero_picks,player_hero_3_win_rate,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate
0,0.7,0.5,8230722475,0.333333,"[Sven, Tiny, Clockwerk, Shadow Demon, Dark See...",0.75,0.600000,0.800000,0.714286,0.80,0.550000,0.600000,0.4,0.769231,0.500000
1,0.5,0.7,8230701740,0.600000,"[Ancient Apparition, Magnus, Pudge, Puck, Sven...",0.70,0.642857,0.300000,0.300000,0.50,0.500000,0.384615,0.4,0.666667,0.550000
2,0.6,0.6,8230693148,0.625000,"[Tinker, Pangolier, Tiny, Leshrac, Lifestealer...",1.00,0.777778,0.588235,0.600000,0.50,0.714286,0.692308,0.5,0.500000,0.833333
3,0.8,0.4,8230677659,0.350000,"[Zeus, Pudge, Wraith King, Sniper, Death Proph...",0.45,0.250000,0.625000,1.000000,0.00,0.350000,0.625000,0.6,0.600000,1.000000
4,0.4,0.7,8230656847,0.600000,"[Jakiro, Tidehunter, Lina, Rubick, Tiny, Anti-...",0.40,0.450000,0.384615,0.500000,0.65,0.538462,0.400000,0.5,0.350000,0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0.5,0.5,5999283181,0.500000,"[Enchantress, Timbersaw, Tiny, Wraith King, Or...",0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96817,0.0,1.0,5999249937,1.000000,"[Void Spirit, Brewmaster, Terrorblade, Snapfir...",1.00,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.000000,0.500000
96818,0.5,0.5,5999214195,0.500000,"[Centaur Warrunner, Hoodwink, Razor, Grimstrok...",0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96819,0.5,0.5,5999201501,0.500000,"[Ancient Apparition, Enchantress, Magnus, Embe...",0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000


In [ ]:
from dota_oracle_pipeline.feature_transformation.feature_encoder import FeatureEncoder
from dota_oracle_common.repositories.heroes_repository import HeroesRepository

async with AsyncSession(engine) as session:
    heros_repo = HeroesRepository(session)
    hero_map = await heros_repo.get_hero_id_map()

hero_df = features_df[["match_id", "hero_picks"]]

encoded_hero_features = FeatureEncoder.encode_hero_features(hero_features=hero_df, hero_map=hero_map)

dota_oracle_common.repositories.heroes_repository - Missing Hero map data


KeyError: "None of [Index(['match_id, hero_picks'], dtype='object')] are in the [columns]"

In [ ]:
encoded_hero_features